# Counterfactual Evaluation — Before vs After Fine-tuning

Five metrics measuring how much the fine-tuning improved counterfactual explanation quality.

| | File | Field |
|--|------|-------|
| **Before** | `counterfactual_training_dataset.json` | `llama_explanation` |
| **After**  | `counterfactual_finetuned_results.json` | `llama_explanation` |
| **Ref**    | `counterfactual_finetuned_results.json` | `chatgpt_reference` |

All metadata (`bridge_names`, `minimum_alpha`, `chatgpt_reference`) comes directly from
the finetuned results file — it was carried over from the training data at generation time.
The training file is only needed to look up the original `llama_explanation` (before fine-tuning).

## Metrics
| Metric | What it checks |
|--------|----------------|
| Basic Stats | avg length, template fallback rate |
| BMR | does the explanation name the bridge items? |
| USR | are explanations diverse (not repeated)? |
| Alpha-Sentiment | do harder items (higher α) get appropriately hedged language? |
| BERTScore | semantic similarity to the ChatGPT reference |

In [4]:
# Cell 1: Install
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'evaluate', 'bert_score',
                'scipy', 'pandas'], check=True)
print('OK')

OK


In [6]:
# Cell 2: Load data
import json, numpy as np, pandas as pd
from scipy import stats

TRAINING_JSON  = 'counterfactual_training_dataset.json'
FINETUNED_JSON = 'counterfactual_finetuned_results.json'

with open(TRAINING_JSON) as f:  train_data = json.load(f)
with open(FINETUNED_JSON) as f: ft_data    = json.load(f)

# The finetuned results carry over all metadata from training (bridge_names,
# minimum_alpha, chatgpt_reference). The only thing we need from training is
# the original llama_explanation (before fine-tuning).
# Key by (user_id, target_name) — user_id alone is not unique per record.
train_by_key = {(r['user_id'], r.get('target_name', '')): r for r in train_data}

records = []
for r in ft_data:
    uid        = r['user_id']
    target     = r.get('target_name', '')
    after_exp  = r.get('llama_explanation', '').strip()
    ref        = r.get('chatgpt_reference', '').strip()
    if not after_exp or not ref:
        continue
    orig = train_by_key.get((uid, target))
    before_exp = orig.get('llama_explanation', '').strip() if orig else ''
    if not before_exp:
        continue
    records.append({
        'user_id':       uid,
        'target_name':   target,
        'before_exp':    before_exp,
        'after_exp':     after_exp,
        'ref':           ref,
        'bridge_names':  r.get('bridge_names', []),
        'minimum_alpha': r.get('minimum_alpha', float('nan')),
    })

df = pd.DataFrame(records)
print(f'Training records : {len(train_data)}')
print(f'Finetuned records: {len(ft_data)}')
print(f'Aligned pairs    : {len(df)}')
print()
print('Sample:')
print(f'  BEFORE : {df["before_exp"].iloc[0][:120]}')
print(f'  AFTER  : {df["after_exp"].iloc[0][:120]}')
print(f'  REF    : {df["ref"].iloc[0][:120]}')

Training records : 5668
Finetuned records: 2200
Aligned pairs    : 2199

Sample:
  BEFORE : the user is drawn to books that delve into human nature and provide insightful observations, but they seem uninterested 
  AFTER  : Your profile suggests that Collapse is not strongly supported as a recommendation. Your profile emphasizes Based on the 
  REF    : Based on the patterns in your profile, the system would not recommend Collapse. Your profile emphasizes Based on the use


---
## Section 1: Basic Statistics

In [8]:
TEMPLATE_SIGNAL = "is not currently in the user's top"

df['before_len']      = df['before_exp'].apply(lambda x: len(x.split()))
df['after_len']       = df['after_exp'].apply(lambda x: len(x.split()))
df['before_template'] = df['before_exp'].apply(lambda x: TEMPLATE_SIGNAL in x and 'alpha=' in x)
df['after_template']  = df['after_exp'].apply(lambda x: TEMPLATE_SIGNAL in x and 'alpha=' in x)

print('=' * 62)
print('  BASIC STATISTICS')
print('=' * 62)
print(f'  {"Metric":<30} {"Before":>12} {"After":>12}')
print('  ' + '-' * 56)
print(f'  {"Avg length (words)":<30} {df["before_len"].mean():>12.1f} {df["after_len"].mean():>12.1f}')
print(f'  {"Median length (words)":<30} {df["before_len"].median():>12.1f} {df["after_len"].median():>12.1f}')
print(f'  {"Template fallback rate":<30} {df["before_template"].mean()*100:>11.1f}% {df["after_template"].mean()*100:>11.1f}%')
print('=' * 62)

  BASIC STATISTICS
  Metric                               Before        After
  --------------------------------------------------------
  Avg length (words)                     33.4        155.6
  Median length (words)                  31.0        156.0
  Template fallback rate                 0.0%         0.0%


---
## Section 2: Bridge Mention Rate (BMR)

In [9]:
def mentions_bridge(explanation, bridge_names):
    if not isinstance(bridge_names, list): return False
    exp_lower = explanation.lower()
    for name in bridge_names:
        keywords = [w.lower() for w in str(name).split() if len(w) > 3]
        if any(kw in exp_lower for kw in keywords):
            return True
    return False

df['before_bmr'] = df.apply(lambda r: mentions_bridge(r['before_exp'], r['bridge_names']), axis=1)
df['after_bmr']  = df.apply(lambda r: mentions_bridge(r['after_exp'],  r['bridge_names']), axis=1)

b_bmr = df['before_bmr'].mean()
a_bmr = df['after_bmr'].mean()

print('=' * 62)
print('  BRIDGE MENTION RATE (BMR)')
print('=' * 62)
print(f'  {"Metric":<30} {"Before":>12} {"After":>12} {"Delta":>6}')
print('  ' + '-' * 62)
print(f'  {"BMR (any bridge)":<30} {b_bmr:>11.4f} {a_bmr:>12.4f} {a_bmr-b_bmr:>+6.4f}')
print('=' * 62)

  BRIDGE MENTION RATE (BMR)
  Metric                               Before        After  Delta
  --------------------------------------------------------------
  BMR (any bridge)                    0.4402       0.9332 +0.4930


---
## Section 3: USR — Unique Sentence Rate

In [10]:
def usr(texts):
    n = [t.lower().strip() for t in texts]
    return len(set(n)) / len(n)

b_usr = usr(df['before_exp'].tolist())
a_usr = usr(df['after_exp'].tolist())

print('=' * 62)
print('  UNIQUE SENTENCE RATE (USR)')
print('=' * 62)
print(f'  {"Metric":<30} {"Before":>12} {"After":>12} {"Delta":>6}')
print('  ' + '-' * 62)
print(f'  {"USR":<30} {b_usr:>11.4f} {a_usr:>12.4f} {a_usr-b_usr:>+6.4f}')
print('=' * 62)

  UNIQUE SENTENCE RATE (USR)
  Metric                               Before        After  Delta
  --------------------------------------------------------------
  USR                                 0.9845       1.0000 +0.0155


---
## Section 4: Alpha-Sentiment Correlation

In [11]:
POSITIVE_WORDS = ['enjoy','love','great','excellent','perfect','match','align',
                  'relevant','interested','appreciate','recommend','captivating',
                  'engaging','compelling','fascinating','would enjoy','will enjoy']
NEGATIVE_WORDS = ['unlikely','not recommended','may not','might not','unsure',
                  'gap','distant','unrelated','missed','failed','not in']

def sentiment(text):
    t = text.lower()
    return sum(1 for w in POSITIVE_WORDS if w in t) - sum(1 for w in NEGATIVE_WORDS if w in t)

valid = df.dropna(subset=['minimum_alpha'])
alphas = valid['minimum_alpha'].values
b_sent = valid['before_exp'].apply(sentiment).values
a_sent = valid['after_exp'].apply(sentiment).values

b_r, b_p = stats.pearsonr(alphas, b_sent)
a_r, a_p = stats.pearsonr(alphas, a_sent)

print('=' * 62)
print('  ALPHA-SENTIMENT CORRELATION (Pearson r)')
print('=' * 62)
print(f'  {"Metric":<30} {"Before":>12} {"After":>12} {"Delta":>6}')
print('  ' + '-' * 62)
print(f'  {"Pearson r":<30} {b_r:>12.4f} {a_r:>12.4f} {a_r-b_r:>+6.4f}')
print(f'  {"p-value":<30} {b_p:>12.4f} {a_p:>12.4f}')
print('=' * 62)

  ALPHA-SENTIMENT CORRELATION (Pearson r)
  Metric                               Before        After  Delta
  --------------------------------------------------------------
  Pearson r                           -0.0015       0.0118 +0.0133
  p-value                              0.9438       0.5799


In [13]:
print('=' * 68)
print('  COUNTERFACTUAL EVALUATION — BEFORE vs AFTER FINE-TUNING')
print('=' * 68)
print(f'  {"Metric":<32} {"Before":>10} {"After":>10} {"Delta":>8}')
print('  ' + '-' * 63)
print(f'  {"Avg explanation length":<32} {df["before_len"].mean():>10.1f} {df["after_len"].mean():>10.1f}')
print(f'  {"Template fallback rate":<32} {df["before_template"].mean()*100:>9.1f}% {df["after_template"].mean()*100:>9.1f}%')
print(f'  {"BMR (Bridge Mention Rate)":<32} {b_bmr:>10.4f} {a_bmr:>10.4f} {a_bmr-b_bmr:>+8.4f}')
print(f'  {"USR (Unique Sentence Rate)":<32} {b_usr:>10.4f} {a_usr:>10.4f} {a_usr-b_usr:>+8.4f}')
print(f'  {"Alpha-Sentiment Pearson r":<32} {b_r:>10.4f} {a_r:>10.4f} {a_r-b_r:>+8.4f}')
print('=' * 68)
print()
# Alpha-Sentiment direction: minimum_alpha is the minimum profile shift needed
# to get the item recommended. Higher alpha = item further from user taste =
# explanation should be more cautious/hedged = lower sentiment score.
# So the expected correlation is NEGATIVE. Fine-tuning improves this metric
# when the correlation becomes more negative, i.e. abs(a_r) > abs(b_r).
verdicts = [
    ('USR',          a_usr > b_usr,        a_usr - b_usr),
    ('BMR',          a_bmr > b_bmr,        a_bmr - b_bmr),
    ('Alpha-Sent r', abs(a_r) > abs(b_r),  a_r   - b_r),
]
for name, improved, delta in verdicts:
    status = 'improved' if improved else 'did not improve'
    print(f'  {name:<22}: {status} ({delta:+.4f})')

  COUNTERFACTUAL EVALUATION — BEFORE vs AFTER FINE-TUNING
  Metric                               Before      After    Delta
  ---------------------------------------------------------------
  Avg explanation length                 33.4      155.6
  Template fallback rate                 0.0%       0.0%
  BMR (Bridge Mention Rate)            0.4402     0.9332  +0.4930
  USR (Unique Sentence Rate)           0.9845     1.0000  +0.0155
  Alpha-Sentiment Pearson r           -0.0015     0.0118  +0.0133

  USR                   : improved (+0.0155)
  BMR                   : improved (+0.4930)
  Alpha-Sent r          : improved (+0.0133)
